# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

First, let's explore the record sets and their main fields. All referencing is done via `@id`.

In [ ]:
# List all available record sets and their IDs
recordsets = list(dataset.metadata.record_sets)
print(f"Found {len(recordsets)} record set(s):")
for idx, rset in enumerate(recordsets):
    print(f"[{idx}] RecordSet @id: {rset['@id']} | name: {rset.get('name', '(no name)')}")

# For the first record set, list its fields and columns
if recordsets:
    first_recordset_id = recordsets[0]["@id"]
    print(f"\nFields in record set '@id': {first_recordset_id}")
    recordset_obj = dataset.metadata.find_record_set(record_set_id=first_recordset_id)
    if recordset_obj and hasattr(recordset_obj, 'fields'):
        for f in recordset_obj.fields:
            print(f"  Field @id: {f['@id']} | name: {f.get('name', 'n/a')} | dataType: {f.get('dataType', 'n/a')}")
    else:
        print("  No fields found in this record set.")
else:
    print('No record sets defined in the metadata.')

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
#--- Determine which record set IDs to process. Here, we use all record sets found above.
record_set_ids = [rs["@id"] for rs in recordsets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  DataFrame shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")
    print()
# Display the head of the first DataFrame, if available
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"Preview of records for record set '@id': {first_id}")
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data, or grouping by key attributes.
We will demonstrate this on the first record set using one numeric field (selected from the overview above), referencing by `@id`.

In [ ]:
import numpy as np

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Try to auto-find a numeric field
    numeric_field_id = None
    recordset_obj = dataset.metadata.find_record_set(record_set_id=record_set_id)
    if recordset_obj and hasattr(recordset_obj, 'fields'):
        for f in recordset_obj.fields:
            if f.get('dataType','').lower() in ['float','integer','number','double']:
                if f['@id'] in df.columns:
                    numeric_field_id = f['@id']
                    break
    # If not found, just use the first column
    if not numeric_field_id and len(df.columns):
        numeric_field_id = df.columns[0]
    print(f"Using numeric field: {numeric_field_id}")

    # Make sure the column is numeric
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    except Exception as e:
        print(f"Warning: could not convert field to numeric: {e}")

    # Filtering
    threshold = df[numeric_field_id].quantile(0.75) # Use 75th percentile as example
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (top 25%): {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by another field (try the second field, if available)
    group_field = None
    possible_fields = [c for c in df.columns if c != numeric_field_id]
    if possible_fields:
        group_field = possible_fields[0]
    if group_field:
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
else:
    print('No record sets to analyze.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id and len(df) > 0:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, show boxplot
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a dataset described by a Croissant schema using `mlcroissant`.
- Explore available record sets and their schema via `@id` referencing.
- Extract record data and perform basic exploratory data analysis, including numeric field selection, filtering, normalization, grouping, and visualization.

You may extend this notebook with further domain-specific analyses or visualizations as needed.